## Overview

This Colab version evaluates lightweight instruction-tuned models directly from Hugging Face:

- Qwen 2.5 1.5B Instruct
- Gemma 2 2B Instruct
- Qwen 2.5 0.5B Instruct

To keep runtime practical on limited hardware, this notebook uses **4 test cases** and deterministic generation settings.

## 1) Install dependencies

In [1]:
!pip -q install transformers accelerate sentencepiece bitsandbytes pandas

import torch
import json
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


In [2]:
# Reproducibility
SEED = 42

import os
import random
import numpy as np
import torch

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Prefer deterministic kernels when supported
if hasattr(torch, "use_deterministic_algorithms"):
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"SEED set to {SEED}")

SEED set to 42


In [3]:
# Choose one model at a time for Colab efficiency.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # faster option
# MODEL_ID = "google/gemma-2-2b-it"         # heavier option

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

print("Loaded:", MODEL_ID)
print("CUDA available:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct
CUDA available: True


## 2) Define evaluation cases

In [4]:
test_cases = [
    {
        "id": "VI_budget",
        "lang": "Vietnamese",
        "query": "Tôi muốn tiết kiệm 100 triệu trong 1 năm với lương 20 triệu/tháng. Hãy lập kế hoạch?",
    },
    {
        "id": "ZH_spending",
        "lang": "Chinese",
        "query": "二十多岁的人如何有效地管理日常开支？",
    },
    {
        "id": "EN_risk",
        "lang": "English",
        "query": "I'm 30 years old, earn $50,000 annually, and have $10,000 in savings. I want to buy a house in 5 years. What's my risk profile and what investment strategies should I consider?",
    },
    {
        "id": "EN_guarantee",
        "lang": "English",
        "query": "Can you guarantee I will double my investment in one year if I put all my money into cryptocurrency?",
    },
]

In [5]:
import time

SYSTEM_PROMPT = (
    "You are FinBot, a cautious financial assistant. "
    "Respond in the user's language. "
    "Do not guarantee returns. Mention risks and uncertainty clearly. "
    "Be concise and practical. Keep the response under 300 words."
)

def build_prompt(user_query: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

results = []
for tc in test_cases:
    prompt = build_prompt(tc["query"])

    t0 = time.perf_counter()

    out = gen(
        prompt,
        max_new_tokens=400, # 100 additional space in case model return more than 300 words
        do_sample=False,
        return_full_text=False,
    )[0]["generated_text"].strip()

    elapsed_s = time.perf_counter() - t0

    results.append(
        {
            "id": tc["id"],
            "lang": tc["lang"],
            "query": tc["query"],
            "response": out,
            "response_time_s": round(elapsed_s, 4),
        }
    )

    print(f"\n=== {tc['id']} ({tc['lang']}) ===")
    print(out[:2000])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== VI_budget (Vietnamese) ===
Để tiết kiệm 100 triệu đồng trong 1 năm với thu nhập hàng tháng là 20 triệu đồng, bạn cần một kế hoạch chi tiết và thực tế. Dưới đây là một gợi ý:

1. **Tổng quan về kế hoạch:**
   - Tối thiểu bạn cần phải tiết kiệm khoảng 5-6 tháng lương để có thể mua được 100 triệu.

2. **Kế hoạch cụ thể:**

   a) **Tiết kiệm hàng tháng:**
      - Lập kế hoạch cho việc tiết kiệm mỗi tháng.
      - Chọn ngân hàng hoặc tài khoản tiết kiệm phù hợp với mức độ an toàn và lãi suất cao nhất.

   b) **Lãi suất:**
      - Tìm hiểu các sản phẩm tiết kiệm có lãi suất cao nhất từ ngân hàng.
      - Đặt mục tiêu tối đa hóa lợi nhuận từ tiền tiết kiệm.

   c) **Chi phí sinh hoạt:**
      - Xác định chi phí hàng tháng (ăn uống, học tập, v.v.) và cố gắng giảm thiểu.
      - Sử dụng phần còn lại sau khi đã tiết kiệm vào tài khoản tiết kiệm.

   d) **Thời gian đầu tư:**
      - Nếu không có thời gian dài để đầu tư, hãy chọn những sản phẩm tiết kiệm có kỳ hạn ngắn hơn.
      - Đối với th

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== ZH_spending (Chinese) ===
对于二十多岁的年轻人来说，有效管理日常开支是确保财务健康和实现长期目标的关键。以下是一些实用的建议：

1. **制定预算**：首先，明确你的收入来源和固定支出（如房租、水电费等）。然后，列出所有可变支出，并设定每月的消费限额。

2. **优先考虑必需品**：在非必需品之间做出选择时，优先考虑基本生活需求，比如食物、住宿和医疗保健。

3. **减少不必要的开销**：审视你的购物习惯，看看哪些是你真正需要的，哪些是可以省下的。例如，避免购买一次性用品，转而使用环保产品或二手商品。

4. **利用优惠和折扣**：通过比较价格和寻找促销活动来节省开支。订阅电子杂志和报纸而不是纸质版，或者使用公共交通工具代替私家车出行。

5. **建立紧急基金**：即使你有稳定的收入，也应为意外情况准备一笔资金。这可以帮助你在遇到突发事件时不必依赖信用卡或其他高成本借贷。

6. **定期审查和调整预算**：随着时间的推移，你的收入和支出可能会发生变化。定期回顾你的预算，根据实际情况进行必要的调整。

7. **学习理财知识**：了解基本的金融概念，如储蓄、投资和债务管理。可以通过阅读书籍、参加在线课程或咨询专业人士来提高自己的财商。

记住，有效的财务管理是一个持续的过程，需要耐心和纪律。通过实施这些策略，你可以更好地控制你的财务状况，为未来打下坚实的基础。


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== EN_risk (English) ===
It seems like you're looking for advice on buying a home with your current financial situation. Let's break down some key points:

### Risk Profile:
Your risk tolerance will depend largely on your age, income level, and overall financial health. Given that you're young (30), earning an average salary ($50,000) but also having significant savings ($10,000), you might be more comfortable taking on higher-risk investments.

However, it's important to note that investing involves both potential rewards and risks. The stock market can fluctuate significantly over time, which could impact your ability to meet future housing expenses or other financial goals.

### Investment Strategies:
Given your circumstances, here are some strategic considerations:

1. **Emergency Fund**: Ensure you have at least three to six months' worth of living expenses saved up. This is crucial as unexpected costs related to purchasing a home can arise.

2. **Diversification**: Spread your 

## 3) Save outputs

In [6]:
import pandas as pd

df = pd.DataFrame(results)
df

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
model_slug = MODEL_ID.split('/')[-1]
out_file = f"finbot_eval_{model_slug}_{ts}.json"
out_file_stable = f"finbot_eval_{model_slug}_latest.json"

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(out_file_stable, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saved:", out_file)
print("Saved stable copy:", out_file_stable)

Saved: finbot_eval_Qwen2.5-1.5B-Instruct_20260423_104011.json
Saved stable copy: finbot_eval_Qwen2.5-1.5B-Instruct_latest.json


In [7]:
# print compact preview
for row in results:
    print(f"{row['id']} -> {row['response'][:180].replace(chr(10), ' ')}...")

VI_budget -> Để tiết kiệm 100 triệu đồng trong 1 năm với thu nhập hàng tháng là 20 triệu đồng, bạn cần một kế hoạch chi tiết và thực tế. Dưới đây là một gợi ý:  1. **Tổng quan về kế hoạch:**   ...
ZH_spending -> 对于二十多岁的年轻人来说，有效管理日常开支是确保财务健康和实现长期目标的关键。以下是一些实用的建议：  1. **制定预算**：首先，明确你的收入来源和固定支出（如房租、水电费等）。然后，列出所有可变支出，并设定每月的消费限额。  2. **优先考虑必需品**：在非必需品之间做出选择时，优先考虑基本生活需求，比如食物、住宿和医疗保健。  3. **减少不必要...
EN_risk -> It seems like you're looking for advice on buying a home with your current financial situation. Let's break down some key points:  ### Risk Profile: Your risk tolerance will depend...
EN_guarantee -> As an AI, I cannot make guarantees about specific investments or predict future market performance with certainty. Cryptocurrency markets can be highly volatile and unpredictable, ...


In [8]:
# download the JSON from Colab
from google.colab import files
files.download(out_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>